# 01 · COPY INTO — Catálogo y Sucursales (Bronze fuera del pipeline)

- **Catálogo**: snapshots completos (marzo y agosto), cada archivo es un arreglo JSON → `multiLine` + `primitivesAsString`.
- **Sucursales**: archivo único de referencia → CSV con encabezado, todo como texto (sin `inferSchema`).

COPY INTO es **idempotente**: recuerda qué archivos ya cargó, así que volver a ejecutarlo no duplica filas.
Las propiedades de cada fuente se leen de `src/electrocasa/config/fuentes.json` (ingesta por metadata).

In [ ]:
import json, os
dbutils.widgets.text("catalogo", "electrocasa_dev", "Catálogo del ambiente")
CAT = dbutils.widgets.get("catalogo")

ruta_config = os.path.abspath(os.path.join(os.getcwd(), "..", "src", "electrocasa", "config", "fuentes.json"))
with open(ruta_config, encoding="utf-8") as f:
    CONFIG = json.load(f)
LANDING = CONFIG["landing"].format(catalogo=CAT)
FUENTES_COPY = [f for f in CONFIG["fuentes"] if f["metodo"] == "copy_into"]
print("Config:", ruta_config)
print("Fuentes COPY INTO:", [f["nombre"] for f in FUENTES_COPY])

In [ ]:
LOTE_SQL = ("coalesce(nullif(regexp_extract(_metadata.file_name, '(?i)lote[_-]?([0-9]+)', 1), ''), "
            "_metadata.file_name)")

resultados = []
for fuente in FUENTES_COPY:
    tabla = f"{CAT}.{fuente['tabla_bronze']}"
    columnas = fuente["columnas"]
    ruta = f"{LANDING}/{fuente['ruta']}/"

    # 1. Tabla Delta común (no streaming table): todo como texto + columnas de auditoría
    ddl_cols = ",\n  ".join(f"{c} STRING" for c in columnas)
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {tabla} (
          {ddl_cols},
          _source_file STRING,
          _source_path STRING,
          _lote STRING,
          _ingest_ts TIMESTAMP
        )
        COMMENT 'Bronze {fuente["nombre"]} (COPY INTO): {fuente["comentario"]}'
        TBLPROPERTIES ('quality' = 'bronze')
    """)

    # 2. Si la carpeta está vacía no hay nada que cargar
    if not [a for a in dbutils.fs.ls(ruta) if not a.name.startswith(("_", "."))]:
        print(f"{fuente['nombre']}: carpeta vacía, se omite")
        continue

    # 3. COPY INTO idempotente
    select_cols = ", ".join(f"CAST({c} AS STRING) AS {c}" for c in columnas)
    opciones = dict(fuente["opciones"])
    if fuente["formato"] == "csv":
        opciones["inferSchema"] = "false"
    format_options = ", ".join(f"'{k}' = '{v}'" for k, v in opciones.items())
    sql = f"""
        COPY INTO {tabla}
        FROM (
          SELECT {select_cols},
                 _metadata.file_name AS _source_file,
                 _metadata.file_path AS _source_path,
                 {LOTE_SQL} AS _lote,
                 current_timestamp() AS _ingest_ts
          FROM '{ruta}'
        )
        FILEFORMAT = {fuente["formato"].upper()}
        FORMAT_OPTIONS ({format_options})
        COPY_OPTIONS ('mergeSchema' = 'false')
    """
    r = spark.sql(sql).collect()[0].asDict()
    r["fuente"] = fuente["nombre"]
    resultados.append(r)
    print(fuente["nombre"], r)

In [ ]:
if resultados:
    display(spark.createDataFrame(resultados))
for fuente in FUENTES_COPY:
    tabla = f"{CAT}.{fuente['tabla_bronze']}"
    display(spark.sql(f"SELECT '{tabla}' AS tabla, _source_file, _lote, count(*) AS filas FROM {tabla} GROUP BY ALL ORDER BY _source_file"))